# TriadLM M1 — real 50M training on corpus_v2 (Kaggle free T4)

Prereqs: **GPU T4 ON**, **Internet ON**, notebook #2 (`m2_corpus`) finished so
`<you>/triadlm-corpus-v2` (dataset) exists on the Hub. ~2h total: run the train
cell, and if the session dies, re-run it with the `--resume` line (checkpoints
save every 500 steps, nothing is lost).

In [ ]:
!pip install -q torch tokenizers pyyaml tqdm "pydantic>=2" requests huggingface_hub
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-ONLY — enable GPU!")

In [ ]:
!git clone https://github.com/PhillipMtalika/triadlm.git
%cd triadlm
!pwd && ls
!python -m pytest tests/test_tokenizer.py tests/test_model.py -q 2>&1 | tail -n 1

In [ ]:
# Pull corpus_v2 shards + manifest + tokenizer from the free Hub dataset (no rebuild).
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import HfApi
me = HfApi(token=token).whoami(token)["name"]
import shutil, os
dst = snapshot_download(repo_id=f"{me}/triadlm-corpus-v2", repo_type="dataset", token=token)
os.makedirs("data/shards/corpus_v2", exist_ok=True)
os.makedirs("data/manifests", exist_ok=True)
os.makedirs("data/checkpoints/base_50m/tokenizer", exist_ok=True)
import glob as _g
n = 0
for f in _g.glob(os.path.join(dst, "*.pt")):
    shutil.copy(f, "data/shards/corpus_v2/")
    n += 1
shutil.copy(os.path.join(dst, "corpus_v2.json"), "data/manifests/corpus_v2.json")
for f in _g.glob(os.path.join(dst, "tokenizer", "*")):
    shutil.copy(f, "data/checkpoints/base_50m/tokenizer/")
print(f"shards: {n}")
!python -c "import torch; tr=torch.load('data/shards/corpus_v2/train-000.pt',weights_only=True); print('tokens/shard:',len(tr))"

In [ ]:
# Train (~2h). If interrupted: uncomment the newest step_*.pt resume line and re-run.
!python -m triadlm.train --config configs/m3_50m.yaml
# !ls data/checkpoints/kaggle_m1/
# !python -m triadlm.train --config configs/m3_50m.yaml --resume data/checkpoints/kaggle_m1/step_3500.pt

In [ ]:
!python -m evals.run_eval --checkpoint data/checkpoints/kaggle_m1/final.pt --variant base --out experiments/runs/kaggle-m1.json
!head -n 3 experiments/registry.csv

In [ ]:
from huggingface_hub import HfApi, create_repo
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)
me = api.whoami(token)["name"]
create_repo(f"{me}/triadlm-m1-50m", private=False, exist_ok=True, token=token)
api.upload_folder(folder_path="data/checkpoints/kaggle_m1", repo_id=f"{me}/triadlm-m1-50m", token=token)
print("uploaded m1")